# Deep learning on images

## Load modules from repo

In [71]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [72]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [73]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.image import load_image, get_image_features_with_hash, get_image_md5_hash

In [74]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.image)

<module 'src.preprocessing.image' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/preprocessing/image.py'>

## Preprocessing

### Load split dataset

In [75]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')

### Encoding of target

Using a LabelEncoder is needed by to_categorical.

In [77]:
y_train.info()

<class 'pandas.core.series.Series'>
Index: 67932 entries, 1887 to 10092
Series name: prdtypecode
Non-Null Count  Dtype
--------------  -----
67932 non-null  int64
dtypes: int64(1)
memory usage: 1.0 MB


In [78]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# le.fit(y_train)

In [79]:
import joblib
path='artifacts/deep_learning_on_images_v1/y_label_encoder.joblib'

In [80]:
# # Save label encoder
# joblib.dump(le, path)

In [81]:
# Load label encoder
le = joblib.load(path)

In [82]:
num_classes=len(le.classes_)
num_classes

27

In [83]:
y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [84]:
# One-hot encoding
from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)

### Encoding of features


In [85]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67932 entries, 1887 to 10092
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   designation             67932 non-null  object 
 1   description             44068 non-null  object 
 2   productid               67932 non-null  int64  
 3   imageid                 67932 non-null  int64  
 4   image_pHash             67932 non-null  object 
 5   gray_image_pHash        67932 non-null  object 
 6   mean_r                  67932 non-null  float64
 7   mean_g                  67932 non-null  float64
 8   mean_b                  67932 non-null  float64
 9   std_r                   67932 non-null  float64
 10  std_g                   67932 non-null  float64
 11  std_b                   67932 non-null  float64
 12  median_r                67932 non-null  float64
 13  median_g                67932 non-null  float64
 14  median_b                67932 non-null  

In [87]:
l=X_train.columns.tolist()
print(l)

['designation', 'description', 'productid', 'imageid', 'image_pHash', 'gray_image_pHash', 'mean_r', 'mean_g', 'mean_b', 'std_r', 'std_g', 'std_b', 'median_r', 'median_g', 'median_b', 'mean_gray', 'std_gray', 'median_gray', 'essential_pixel_count', 'x_min', 'y_min', 'x_max', 'y_max', 'image_hash_md5', 'len_designation', 'len_description', 'essential_width', 'essential_height', 'essential_aspect_ratio', 'essential_area', 'rectangleness', 'image_folder']


In [88]:
columns_to_drop=['designation', 'description', 'gray_image_pHash']

In [89]:
for c in columns_to_drop:
    l.remove(c)
print(l)

['productid', 'imageid', 'image_pHash', 'mean_r', 'mean_g', 'mean_b', 'std_r', 'std_g', 'std_b', 'median_r', 'median_g', 'median_b', 'mean_gray', 'std_gray', 'median_gray', 'essential_pixel_count', 'x_min', 'y_min', 'x_max', 'y_max', 'image_hash_md5', 'len_designation', 'len_description', 'essential_width', 'essential_height', 'essential_aspect_ratio', 'essential_area', 'rectangleness', 'image_folder']


In [90]:
columns_for_standard_scaler=['mean_r', 'mean_g', 'mean_b', 'std_r', 'std_g', 'std_b', 'median_r', 'median_g', 'median_b', 'mean_gray', 'std_gray', 'median_gray', 'essential_pixel_count', 'x_min', 'y_min', 'x_max', 'y_max', 'len_designation', 'len_description', 'essential_width', 'essential_height', 'essential_aspect_ratio', 'essential_area', 'rectangleness']

In [91]:
for c in columns_for_standard_scaler:
    l.remove(c)
print(l)

['productid', 'imageid', 'image_pHash', 'image_hash_md5', 'image_folder']


In [92]:
columns_for_label_encoding=['image_pHash', 'image_hash_md5']

In [93]:
for c in columns_for_label_encoding:
    l.remove(c)
print(l)

['productid', 'imageid', 'image_folder']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [ ]:
tabular_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
    ],
    remainder='passthrough' # Garde les autres colonnes si besoin
)

In [ ]:
def get_image_path(row, prefix='Dataset/images/image_train'): # FIXME image_folder doit toujours être train !
    path = f"{prefix}/image_{row.imageid}_product_{row.productid}.jpg"
    return path

## Model creation

In [6]:
import tensorflow

2025-09-29 10:53:39.900607: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-29 10:53:40.390395: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-29 10:53:41.781366: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [7]:
print("Num GPUs Available: ", len(tensorflow.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [8]:
print(f"tensorflow: {tensorflow.__version__}")

tensorflow: 2.20.0


In [ ]:
# For CNN
from tensorflow.keras.layers import Input, Conv2D, Dense, Dropout, Flatten, MaxPooling2D, Rescaling
from tensorflow.keras.models import Model

In [5]:
inputs = Input(shape=(500, 500, 3), name="Input")

## Training

### Saving and loading a model

In [ ]:
# # Callback pour sauvegarder le meilleur modèle au fur et à mesure
# save = ModelCheckpoint(
#     'best_model.h5',
#     save_best_only=True,
#     monitor='val_accuracy',
#     mode='max'
# )
# model_history = model.fit(train_ds,
#                           validation_data=val_ds,
#                           epochs=50,
#                           callbacks = [save])

In [ ]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save('mon_modele.keras')

In [ ]:

# #    Pour charger un modèle sauvegardé on utilise la fonction load_model() de tensorflow.keras.models.
# model_loaded = load_model('best_model.keras')

## Evaluation